In [42]:
import pandas as pd
import MaxNLP
from pathlib import Path
from tqdm.auto import tqdm
import json
tqdm.pandas(desc="Applying")
%load_ext autoreload
%autoreload 2
    
from datetime import date
date_str = date.today().isoformat()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import re
#df=pd.read_json("clean_df.json")
df_t=pd.read_json("2025-03-03 Sample25_translated.json")

# replace whitespace with single whitespace
cols=["QID8"]
df_t[cols] = df_t[cols].map(lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else "")


In [6]:
# Load the Nebula Client
from dotenv import load_dotenv
import openAI_key

client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
available_models = MaxNLP.get_nebula_models(client)

print(f"Models: {available_models}\n\n")


Nebula client loaded
Models: ['deepseek-r1:8b', 'deepseek-r1:1.5b', 'SURF.default-text-large', 'FAST.gemma3:12b', 'FAST.gpt-oss:120b', 'FAST.gpt-oss:20b', 'go-assist', 'research-railway-guide', 'SURF.Qwen 2.5 Coder 32B Instruct AWQ', 'SURF.Qwen 2.5 VL 32B Instruct AWQ', 'vu-rdm-support-chatbot---test', 'translategemma:12b', 'SURF.Mistral-Small-3.2-24B-Instruct-2506', 'SURF.Qwen3.5 122B A10B NVFP4', 'SURF.gpt-oss-120b', 'llama3.1:8b']




In [63]:
system_prompt="""
You are a professional translator. 
Translate the following citizen survey response to English
1. Preserve text structure, meaning, tone, and any informal style.
2. Do not interpret, explain, or comment.
3. Do not include any headings, quotes, or extra formatting.
4. Output ONLY the translation."""

def LLM_translate_text(text, model="FAST.gpt-oss:120b", prompt=system_prompt):
    user_text = f"Translate the following text to English:\n{text}"

    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_text},
        ],
    )

    return resp.choices[0].message.content.strip()

df_t["QID8_en"] = df_t["QID8"].progress_apply(LLM_translate_text)


Applying:   0%|          | 0/250 [00:00<?, ?it/s]

In [95]:
df_t.to_json("2026-05-02 new translated.json")

# create sample sheet for manual Annotation

In [44]:
codebook_causes = Path("codebooks/2026-05-06 causes codebook.md").read_text(encoding="utf-8")
codebook_causes_json=MaxNLP.markdown_to_json(codebook_causes)
codes_causes= [i["label"] for i in codebook_causes_json]
codes_causes

['EU political actors',
 'National political actors',
 'Industry and market actors',
 'Media actors',
 "Russia's war on Ukraine",
 'Too-fast energy transition',
 'Dependence on fossil fuels',
 'Concentrated energy import dependency',
 'Uncontrollable natural disasters',
 'External shocks and technical failures',
 'Abstract societal and ideological explanations',
 'EU Sanctions against Russia',
 'nuclear phase-out / reduced national production',
 'Insufficient price regulation']

In [45]:
import pandas as pd

n = len(df_t)
cols =["labels"]+ df_t.index.tolist()

codes=codes_causes
annotation_df = pd.DataFrame(
    [[c] + [0]*n for c in codes] +
    [[""] + [""]*n] +
    [["text"] + df_t["QID8"].tolist()] +
    [["translation"] + df_t["QID8_en"].tolist()],
    columns=cols
)

annotation_df.to_excel("2026-05-14 causes_manual_annotation.xlsx", index=False, freeze_panes=(1, 1))

In [17]:
#QID8 == Causes


5813    les riches et les industrielles, ceux de tout ...
8998    Izrael który bezprawnie okupuje tereny Palesty...
6937    Les fluctuations des marchés des énergies qui ...
1343    Is eigelijk niets nieuws temeer dat nooit op t...
7283    C'est le debut de la guerre en Ukraine qui a f...
                              ...                        
6614    Depender demasiado que energías procedentes de...
776     Los políticos corruptos y ladrones de mierda y...
5455    No nos dicen la verdad , nos tienen engañados ...
9423    Con la crisis de la guerra de Rusia y Ucrania ...
778     El cambio a energía solar y eólica sin otras a...
Name: QID8, Length: 250, dtype: object